
# 04B. Modelos não lineares — 14 features reduzidas e validação temporal

Modelos avaliados:

- Random Forest
- Extra Trees
- HistGradientBoosting

O notebook usa um conjunto **reduzido de 14 features**, derivado da análise de correlação, informação mútua, estabilidade temporal e redundância das 14 features reduzidas do notebook 03B/04.

## Estratégia de avaliação

Os 8 contratos mais recentes permanecem como **teste final** e não participam da seleção.

Os 40 contratos anteriores formam o conjunto de desenvolvimento. Eles são ordenados pela data de conclusão definitiva e avaliados em quatro janelas temporais expanding:

- Fold 1: 20 contratos para treino → 5 contratos seguintes para validação
- Fold 2: 25 → 5
- Fold 3: 30 → 5
- Fold 4: 35 → 5

Isso permite selecionar o modelo pela consistência ao longo do tempo, em vez de depender de uma única validação pequena.

## Tarefas

**Regressão**
- prever `target_dias_atraso`;
- seleção por menor MAE ponderada média nas janelas.

**Classificação**
- prever `target_atraso`;
- somente snapshots com `flag_apto_classificacao = 1`;
- seleção por maior ROC-AUC média nas janelas;
- threshold escolhido usando somente previsões out-of-fold das janelas temporais.

O teste final é avaliado apenas depois da seleção.


In [1]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (
    ExtraTreesClassifier,
    ExtraTreesRegressor,
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42


## 1. Dataset enriquecido e diretórios

In [2]:

def find_project_root():
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data" / "processed").exists():
            return candidate

    return current


PROJECT_ROOT = find_project_root()


def first_existing(paths):
    for path in paths:
        if path.exists():
            return path.resolve()

    return None


DATASET_PATH = first_existing([
    PROJECT_ROOT / "data/processed/dataset_modelagem_obras_snapshots_enriquecido.csv",
    PROJECT_ROOT / "modelagem_outputs/processed/dataset_modelagem_obras_snapshots_enriquecido.csv",
    PROJECT_ROOT / "dataset_modelagem_obras_snapshots_enriquecido.csv",
    Path("/mnt/data/modelagem_outputs/processed/dataset_modelagem_obras_snapshots_enriquecido.csv"),
    Path("/mnt/data/data/processed/dataset_modelagem_obras_snapshots_enriquecido.csv"),
])

if DATASET_PATH is None:
    raise FileNotFoundError(
        "dataset_modelagem_obras_snapshots_enriquecido.csv não encontrado."
    )


OUTPUT_DIR = Path("/mnt/data/04B_modelos_arvores_features_14_outputs")
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Dataset:", DATASET_PATH)
print("Saídas:", OUTPUT_DIR.resolve())


Dataset: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\processed\dataset_modelagem_obras_snapshots_enriquecido.csv
Saídas: C:\mnt\data\04B_modelos_arvores_features_14_outputs


## 2. Carga e definição das 14 features reduzidas

In [3]:

df = pd.read_csv(
    DATASET_PATH,
    low_memory=False,
    encoding="utf-8-sig",
)

date_columns = [
    "data_corte",
    "data_os",
    "data_termino_original",
    "data_termino_vigente_corte",
    "data_conclusao_definitiva",
    "data_conclusao_controle",
    "data_termino_vigente_final_controle",
]

for column in date_columns:
    if column in df.columns:
        df[column] = pd.to_datetime(
            df[column],
            errors="coerce",
        )


NUMERIC_FEATURES = [
    # Características do contrato
    "prazo_original_dias",
    "perc_desconto_modelo",

    # Evolução temporal
    "pct_prazo_consumido",
    "dias_restantes_prazo_vigente",

    # Aditivos
    "pct_dias_aditados_sobre_prazo_original",
    "qtd_aditivos_valor",

    # Execução física
    "pct_parcelas_atrasadas",
    "dias_desde_ultima_parcela_concluida",

    # Execução financeira
    "pct_financeiro_liberado",
    "atraso_medio_liberacao_dias",

    # Gaps
    "gap_prazo_execucao",
    "gap_prazo_financeiro",
]

CATEGORICAL_FEATURES = [
    "classificacao_contrato",
    "id_setor_divisao",
]

FEATURES = (
    NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
)

FEATURES_REMOVIDAS = [
    "log_valor_total_contrato",
    "dias_assinatura_ate_os",
    "fornecedor_contratos_anteriores",
    "predio_contratos_anteriores",
    "dias_desde_os",
    "qtd_aditivos_prazo",
    "pct_parcelas_concluidas_vs_previstas",
    "atraso_medio_parcelas_dias",
]

assert len(FEATURES) == 14
assert len(set(FEATURES)) == len(FEATURES)


missing = [
    column
    for column in FEATURES
    if column not in df.columns
]

if missing:
    raise ValueError(
        f"Features reduzidas ausentes no dataset: {missing}"
    )


required = [
    "id_contrato",
    "data_corte",
    "data_conclusao_definitiva",
    "target_dias_atraso",
    "target_atraso",
    "flag_apto_treino",
    "flag_apto_classificacao",
    "sample_weight_contrato",
    "sample_weight_classificacao",
    "split_contrato_temporal",
    "split_modelagem",
    "dias_restantes_prazo_vigente",
]

missing_required = [
    column
    for column in required
    if column not in df.columns
]

if missing_required:
    raise ValueError(
        f"Colunas obrigatórias ausentes: {missing_required}"
    )


print("Shape:", df.shape)
print("Contratos:", df["id_contrato"].nunique())
print("Features:", len(FEATURES))
print()
print(FEATURES)


Shape: (633, 109)
Contratos: 48
Features: 14

['prazo_original_dias', 'perc_desconto_modelo', 'pct_prazo_consumido', 'dias_restantes_prazo_vigente', 'pct_dias_aditados_sobre_prazo_original', 'qtd_aditivos_valor', 'pct_parcelas_atrasadas', 'dias_desde_ultima_parcela_concluida', 'pct_financeiro_liberado', 'atraso_medio_liberacao_dias', 'gap_prazo_execucao', 'gap_prazo_financeiro', 'classificacao_contrato', 'id_setor_divisao']


## 3. Separação desenvolvimento × teste final

In [4]:

reg_all = df[
    df["flag_apto_treino"].eq(1)
].copy()

clf_all = df[
    df["flag_apto_classificacao"].eq(1)
].copy()


# Uma linha de referência por contrato para ordenar temporalmente.
contract_table = (
    reg_all[
        [
            "id_contrato",
            "data_conclusao_definitiva",
            "target_atraso",
            "target_dias_atraso",
            "split_contrato_temporal",
        ]
    ]
    .drop_duplicates(
        "id_contrato"
    )
    .sort_values(
        [
            "data_conclusao_definitiva",
            "id_contrato",
        ]
    )
    .reset_index(drop=True)
)


test_contracts = (
    contract_table[
        contract_table[
            "split_contrato_temporal"
        ].eq("test")
    ]["id_contrato"]
    .tolist()
)

development_contracts = (
    contract_table[
        ~contract_table[
            "id_contrato"
        ].isin(test_contracts)
    ]["id_contrato"]
    .tolist()
)


assert len(test_contracts) == 8
assert len(development_contracts) == 40
assert not (
    set(test_contracts)
    & set(development_contracts)
)


dev_contract_table = (
    contract_table[
        contract_table[
            "id_contrato"
        ].isin(development_contracts)
    ]
    .sort_values(
        [
            "data_conclusao_definitiva",
            "id_contrato",
        ]
    )
    .reset_index(drop=True)
)


print(
    "Contratos desenvolvimento:",
    len(development_contracts),
)

print(
    "Contratos teste final:",
    len(test_contracts),
)

print()
print(
    "Teste final:",
    test_contracts,
)

display(
    contract_table.groupby(
        "split_contrato_temporal"
    ).agg(
        contratos=("id_contrato", "nunique"),
        atrasados=("target_atraso", "sum"),
        taxa_atraso=("target_atraso", "mean"),
        media_atraso_dias=("target_dias_atraso", "mean"),
    )
)


Contratos desenvolvimento: 40
Contratos teste final: 8

Teste final: [173, 201, 287, 283, 342, 317, 254, 377]


,contratos,atrasados,taxa_atraso,media_atraso_dias
split_contrato_temporal,,,,
test,8,2,0.2500,21.8750
train,33,16,0.4848,39.0303
validation,7,2,0.2857,12.7143



## 4. Janelas temporais expanding

As janelas são definidas **no nível do contrato**, nunca no nível do snapshot.

Todos os snapshots de um contrato pertencem sempre ao mesmo lado da divisão.


In [5]:

MIN_TRAIN_CONTRACTS = 20
VALIDATION_BLOCK_SIZE = 5
N_FOLDS = 4

folds = []

for fold_number in range(N_FOLDS):
    train_end = (
        MIN_TRAIN_CONTRACTS
        + fold_number
        * VALIDATION_BLOCK_SIZE
    )

    validation_start = train_end
    validation_end = (
        validation_start
        + VALIDATION_BLOCK_SIZE
    )

    train_ids = (
        dev_contract_table
        .iloc[:train_end][
            "id_contrato"
        ]
        .tolist()
    )

    validation_ids = (
        dev_contract_table
        .iloc[
            validation_start:
            validation_end
        ][
            "id_contrato"
        ]
        .tolist()
    )

    folds.append({
        "fold": fold_number + 1,
        "train_ids": train_ids,
        "validation_ids": validation_ids,
    })


fold_summary_rows = []

for fold in folds:
    train_ref = (
        dev_contract_table[
            dev_contract_table[
                "id_contrato"
            ].isin(
                fold["train_ids"]
            )
        ]
    )

    val_ref = (
        dev_contract_table[
            dev_contract_table[
                "id_contrato"
            ].isin(
                fold[
                    "validation_ids"
                ]
            )
        ]
    )

    fold_summary_rows.append({
        "fold": fold["fold"],
        "contratos_treino": len(
            fold["train_ids"]
        ),
        "contratos_validacao": len(
            fold["validation_ids"]
        ),
        "inicio_validacao": (
            val_ref[
                "data_conclusao_definitiva"
            ].min()
        ),
        "fim_validacao": (
            val_ref[
                "data_conclusao_definitiva"
            ].max()
        ),
        "atrasados_treino": int(
            train_ref[
                "target_atraso"
            ].sum()
        ),
        "atrasados_validacao": int(
            val_ref[
                "target_atraso"
            ].sum()
        ),
        "taxa_atraso_validacao": (
            val_ref[
                "target_atraso"
            ].mean()
        ),
    })


fold_summary = pd.DataFrame(
    fold_summary_rows
)

display(fold_summary)


# Sanity checks temporais.
for fold in folds:
    train_ref = (
        dev_contract_table[
            dev_contract_table[
                "id_contrato"
            ].isin(
                fold["train_ids"]
            )
        ]
    )

    val_ref = (
        dev_contract_table[
            dev_contract_table[
                "id_contrato"
            ].isin(
                fold[
                    "validation_ids"
                ]
            )
        ]
    )

    assert (
        train_ref[
            "data_conclusao_definitiva"
        ].max()
        <
        val_ref[
            "data_conclusao_definitiva"
        ].min()
    )

    assert not (
        set(
            fold["train_ids"]
        )
        & set(
            fold[
                "validation_ids"
            ]
        )
    )


,fold,contratos_treino,contratos_validacao,inicio_validacao,fim_validacao,atrasados_treino,atrasados_validacao,taxa_atraso_validacao
0,1,20,5,2021-05-27,2023-07-09,9,3,0.6000
1,2,25,5,2023-07-10,2024-04-23,12,2,0.4000
2,3,30,5,2024-05-06,2024-09-02,14,2,0.4000
3,4,35,5,2024-09-05,2025-03-17,16,2,0.4000



## 5. Preprocessamento para árvores

Mantemos o clipping robusto do 03B, aprendido apenas no treino de cada fold.

Não usamos scaler, porque árvores não dependem da escala das variáveis.


In [6]:

class QuantileClipper(
    BaseEstimator,
    TransformerMixin,
):
    def __init__(
        self,
        lower=0.01,
        upper=0.99,
    ):
        self.lower = lower
        self.upper = upper

    def fit(
        self,
        X,
        y=None,
    ):
        arr = np.asarray(
            X,
            dtype=float,
        )

        self.lower_bounds_ = np.nanquantile(
            arr,
            self.lower,
            axis=0,
        )

        self.upper_bounds_ = np.nanquantile(
            arr,
            self.upper,
            axis=0,
        )

        return self

    def transform(
        self,
        X,
    ):
        arr = np.asarray(
            X,
            dtype=float,
        ).copy()

        return np.clip(
            arr,
            self.lower_bounds_,
            self.upper_bounds_,
        )

    def get_feature_names_out(
        self,
        input_features=None,
    ):
        if input_features is None:
            return np.array(
                [
                    f"x{i}"
                    for i in range(
                        len(self.lower_bounds_)
                    )
                ],
                dtype=object,
            )

        return np.asarray(
            input_features,
            dtype=object,
        )


numeric_pipeline = Pipeline([
    (
        "clipper",
        QuantileClipper(
            lower=0.01,
            upper=0.99,
        ),
    ),
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True,
        ),
    ),
])

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent",
        ),
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        ),
    ),
])


preprocessing = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            NUMERIC_FEATURES,
        ),
        (
            "categorical",
            categorical_pipeline,
            CATEGORICAL_FEATURES,
        ),
    ],
    remainder="drop",
)


## 6. Modelos regularizados

In [7]:

def classification_models():
    return {
        "RandomForest": Pipeline([
            (
                "preprocessing",
                clone(preprocessing),
            ),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    max_depth=5,
                    min_samples_leaf=5,
                    max_features="sqrt",
                    class_weight="balanced_subsample",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]),

        "ExtraTrees": Pipeline([
            (
                "preprocessing",
                clone(preprocessing),
            ),
            (
                "model",
                ExtraTreesClassifier(
                    n_estimators=300,
                    max_depth=5,
                    min_samples_leaf=5,
                    max_features="sqrt",
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]),

        "HistGradientBoosting": Pipeline([
            (
                "preprocessing",
                clone(preprocessing),
            ),
            (
                "model",
                HistGradientBoostingClassifier(
                    learning_rate=0.03,
                    max_iter=200,
                    max_leaf_nodes=7,
                    min_samples_leaf=15,
                    l2_regularization=2.0,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]),
    }


def regression_models():
    return {
        "RandomForest": Pipeline([
            (
                "preprocessing",
                clone(preprocessing),
            ),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=300,
                    max_depth=5,
                    min_samples_leaf=5,
                    max_features="sqrt",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]),

        "ExtraTrees": Pipeline([
            (
                "preprocessing",
                clone(preprocessing),
            ),
            (
                "model",
                ExtraTreesRegressor(
                    n_estimators=300,
                    max_depth=5,
                    min_samples_leaf=5,
                    max_features="sqrt",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]),

        "HistGradientBoosting": Pipeline([
            (
                "preprocessing",
                clone(preprocessing),
            ),
            (
                "model",
                HistGradientBoostingRegressor(
                    loss="absolute_error",
                    learning_rate=0.03,
                    max_iter=200,
                    max_leaf_nodes=7,
                    min_samples_leaf=15,
                    l2_regularization=2.0,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]),
    }


# Parte A — Regressão com validação temporal

In [8]:

def regression_metrics(
    y_true,
    y_pred,
    sample_weight,
):
    y_true = np.asarray(
        y_true,
        dtype=float,
    )

    y_pred = np.maximum(
        0,
        np.asarray(
            y_pred,
            dtype=float,
        ),
    )

    return {
        "mae": mean_absolute_error(
            y_true,
            y_pred,
            sample_weight=sample_weight,
        ),
        "rmse": np.sqrt(
            mean_squared_error(
                y_true,
                y_pred,
                sample_weight=sample_weight,
            )
        ),
        "r2": r2_score(
            y_true,
            y_pred,
            sample_weight=sample_weight,
        ),
    }


reg_cv_rows = []

for fold in folds:
    fold_number = fold["fold"]

    train_part = reg_all[
        reg_all[
            "id_contrato"
        ].isin(
            fold["train_ids"]
        )
    ].copy()

    val_part = reg_all[
        reg_all[
            "id_contrato"
        ].isin(
            fold[
                "validation_ids"
            ]
        )
    ].copy()

    X_train = train_part[
        FEATURES
    ]

    y_train = train_part[
        "target_dias_atraso"
    ].astype(float)

    w_train = train_part[
        "sample_weight_contrato"
    ].astype(float)

    X_val = val_part[
        FEATURES
    ]

    y_val = val_part[
        "target_dias_atraso"
    ].astype(float)

    w_val = val_part[
        "sample_weight_contrato"
    ].astype(float)

    for model_name, model in regression_models().items():
        model.fit(
            X_train,
            y_train,
            model__sample_weight=w_train,
        )

        pred = np.maximum(
            0,
            model.predict(
                X_val
            ),
        )

        reg_cv_rows.append({
            "fold": fold_number,
            "modelo": model_name,
            "contratos_treino": (
                train_part[
                    "id_contrato"
                ].nunique()
            ),
            "contratos_validacao": (
                val_part[
                    "id_contrato"
                ].nunique()
            ),
            **regression_metrics(
                y_val,
                pred,
                w_val,
            ),
        })


reg_cv = pd.DataFrame(
    reg_cv_rows
)

display(
    reg_cv.sort_values(
        [
            "fold",
            "mae",
        ]
    )
)


reg_cv_summary = (
    reg_cv
    .groupby(
        "modelo",
        as_index=False,
    )
    .agg(
        mae_media=("mae", "mean"),
        mae_dp=("mae", "std"),
        rmse_medio=("rmse", "mean"),
        r2_medio=("r2", "mean"),
        folds=("fold", "nunique"),
    )
    .sort_values(
        [
            "mae_media",
            "rmse_medio",
        ]
    )
)


display(reg_cv_summary)


BEST_REGRESSION_MODEL = (
    reg_cv_summary
    .iloc[0][
        "modelo"
    ]
)


print(
    "Melhor regressão pela validação temporal:",
    BEST_REGRESSION_MODEL,
)


,fold,modelo,contratos_treino,contratos_validacao,mae,rmse,r2
2,1,HistGradientBoosting,20,5,30.0000,47.2821,-0.6739
1,1,ExtraTrees,20,5,36.6905,44.0216,-0.4510
0,1,RandomForest,20,5,47.7254,54.7835,-1.2471
5,2,HistGradientBoosting,25,5,63.8000,126.8109,-0.3389
3,2,RandomForest,25,5,65.6771,108.3386,0.0228
4,2,ExtraTrees,25,5,66.9175,109.9894,-0.0073
8,3,HistGradientBoosting,30,5,9.4000,15.4208,-0.5913
6,3,RandomForest,30,5,45.1241,51.3262,-16.6283
7,3,ExtraTrees,30,5,46.1767,49.9742,-15.7118
11,4,HistGradientBoosting,35,5,17.8000,34.1204,-0.3739


,modelo,mae_media,mae_dp,rmse_medio,r2_medio,folds
1,HistGradientBoosting,30.2500,23.9123,55.9085,-0.4945,4
0,ExtraTrees,52.8040,13.8707,68.5478,-5.2467,4
2,RandomForest,56.0768,11.1958,73.5856,-6.0964,4


Melhor regressão pela validação temporal: HistGradientBoosting


# Parte B — Classificação com validação temporal

In [9]:

def safe_roc_auc(
    y_true,
    prob,
    sample_weight=None,
):
    if pd.Series(
        y_true
    ).nunique() < 2:
        return np.nan

    return roc_auc_score(
        y_true,
        prob,
        sample_weight=sample_weight,
    )


def safe_pr_auc(
    y_true,
    prob,
    sample_weight=None,
):
    if pd.Series(
        y_true
    ).nunique() < 2:
        return np.nan

    return average_precision_score(
        y_true,
        prob,
        sample_weight=sample_weight,
    )


def classification_metrics(
    y_true,
    prob,
    threshold,
    sample_weight,
):
    prob = np.asarray(
        prob,
        dtype=float,
    )

    pred = (
        prob >= threshold
    ).astype(int)

    return {
        "roc_auc": safe_roc_auc(
            y_true,
            prob,
            sample_weight,
        ),
        "pr_auc": safe_pr_auc(
            y_true,
            prob,
            sample_weight,
        ),
        "brier": brier_score_loss(
            y_true,
            prob,
            sample_weight=sample_weight,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            pred,
            sample_weight=sample_weight,
        ),
        "precision": precision_score(
            y_true,
            pred,
            sample_weight=sample_weight,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            pred,
            sample_weight=sample_weight,
            zero_division=0,
        ),
        "f1": f1_score(
            y_true,
            pred,
            sample_weight=sample_weight,
            zero_division=0,
        ),
    }


clf_cv_rows = []
oof_rows = []

for fold in folds:
    fold_number = fold["fold"]

    train_part = clf_all[
        clf_all[
            "id_contrato"
        ].isin(
            fold["train_ids"]
        )
    ].copy()

    val_part = clf_all[
        clf_all[
            "id_contrato"
        ].isin(
            fold[
                "validation_ids"
            ]
        )
    ].copy()

    X_train = train_part[
        FEATURES
    ]

    y_train = train_part[
        "target_atraso"
    ].astype(int)

    w_train = train_part[
        "sample_weight_classificacao"
    ].astype(float)

    X_val = val_part[
        FEATURES
    ]

    y_val = val_part[
        "target_atraso"
    ].astype(int)

    w_val = val_part[
        "sample_weight_classificacao"
    ].astype(float)

    if y_train.nunique() < 2:
        raise ValueError(
            f"Fold {fold_number}: treino possui apenas uma classe."
        )

    if y_val.nunique() < 2:
        raise ValueError(
            f"Fold {fold_number}: validação possui apenas uma classe."
        )

    for model_name, model in classification_models().items():
        model.fit(
            X_train,
            y_train,
            model__sample_weight=w_train,
        )

        prob = (
            model.predict_proba(
                X_val
            )[:, 1]
        )

        metrics = classification_metrics(
            y_val,
            prob,
            threshold=0.50,
            sample_weight=w_val,
        )

        clf_cv_rows.append({
            "fold": fold_number,
            "modelo": model_name,
            "contratos_treino": (
                train_part[
                    "id_contrato"
                ].nunique()
            ),
            "contratos_validacao": (
                val_part[
                    "id_contrato"
                ].nunique()
            ),
            **metrics,
        })

        fold_oof = val_part[
            [
                "id_contrato",
                "data_corte",
                "target_atraso",
                "sample_weight_classificacao",
                "dias_restantes_prazo_vigente",
            ]
        ].copy()

        fold_oof[
            "fold"
        ] = fold_number

        fold_oof[
            "modelo"
        ] = model_name

        fold_oof[
            "prob_atraso"
        ] = prob

        oof_rows.append(
            fold_oof
        )


clf_cv = pd.DataFrame(
    clf_cv_rows
)

oof_predictions = pd.concat(
    oof_rows,
    ignore_index=True,
)


display(
    clf_cv.sort_values(
        [
            "fold",
            "roc_auc",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


clf_cv_summary = (
    clf_cv
    .groupby(
        "modelo",
        as_index=False,
    )
    .agg(
        roc_auc_media=("roc_auc", "mean"),
        roc_auc_dp=("roc_auc", "std"),
        pr_auc_media=("pr_auc", "mean"),
        brier_medio=("brier", "mean"),
        balanced_accuracy_media=(
            "balanced_accuracy",
            "mean",
        ),
        folds=("fold", "nunique"),
    )
    .sort_values(
        [
            "roc_auc_media",
            "pr_auc_media",
            "brier_medio",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
)


display(clf_cv_summary)


BEST_CLASSIFICATION_MODEL = (
    clf_cv_summary
    .iloc[0][
        "modelo"
    ]
)


print(
    "Melhor classificação pela validação temporal:",
    BEST_CLASSIFICATION_MODEL,
)


,fold,modelo,contratos_treino,contratos_validacao,roc_auc,pr_auc,brier,balanced_accuracy,precision,recall,f1
2,1,HistGradientBoosting,20,5,0.3406,0.5335,0.4401,0.4958,0.5930,0.2833,0.3835
0,1,RandomForest,20,5,0.3300,0.5213,0.3285,0.3708,0.4681,0.3667,0.4112
1,1,ExtraTrees,20,5,0.3260,0.5183,0.3269,0.4333,0.5238,0.3667,0.4314
4,2,ExtraTrees,25,5,0.4860,0.3671,0.2377,0.5000,0.0000,0.0000,0.0000
3,2,RandomForest,25,5,0.4284,0.3449,0.2475,0.4378,0.1513,0.0455,0.0699
5,2,HistGradientBoosting,25,5,0.2341,0.2842,0.4485,0.3242,0.1852,0.1818,0.1835
7,3,ExtraTrees,30,5,0.8655,0.8348,0.1899,0.5833,1.0000,0.1667,0.2857
6,3,RandomForest,30,5,0.6645,0.5559,0.2294,0.5417,0.5000,0.2500,0.3333
8,3,HistGradientBoosting,30,5,0.3475,0.3421,0.3471,0.4084,0.3165,0.4167,0.3598
10,4,ExtraTrees,35,5,0.3333,0.3339,0.4767,0.2500,0.2500,0.5000,0.3333


,modelo,roc_auc_media,roc_auc_dp,pr_auc_media,brier_medio,balanced_accuracy_media,folds
0,ExtraTrees,0.5027,0.2529,0.5135,0.3078,0.4417,4
2,RandomForest,0.4306,0.1654,0.4346,0.3177,0.4001,4
1,HistGradientBoosting,0.3000,0.0540,0.3670,0.4511,0.3835,4


Melhor classificação pela validação temporal: ExtraTrees



## 7. Threshold usando apenas previsões out-of-fold

O threshold não é escolhido no teste final.

Ele é selecionado pelas previsões de validação das quatro janelas temporais do modelo vencedor.


In [10]:

selected_oof = (
    oof_predictions[
        oof_predictions[
            "modelo"
        ].eq(
            BEST_CLASSIFICATION_MODEL
        )
    ]
    .copy()
)


threshold_rows = []

for threshold in np.arange(
    0.05,
    0.951,
    0.01,
):
    metrics = classification_metrics(
        selected_oof[
            "target_atraso"
        ].astype(int),
        selected_oof[
            "prob_atraso"
        ].astype(float),
        threshold=threshold,
        sample_weight=selected_oof[
            "sample_weight_classificacao"
        ].astype(float),
    )

    threshold_rows.append({
        "threshold": threshold,
        **metrics,
    })


threshold_table = pd.DataFrame(
    threshold_rows
)


BEST_THRESHOLD = float(
    threshold_table
    .sort_values(
        [
            "balanced_accuracy",
            "f1",
            "precision",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .iloc[0][
        "threshold"
    ]
)


print(
    "Threshold escolhido via OOF temporal:",
    round(
        BEST_THRESHOLD,
        2,
    ),
)

display(
    threshold_table.sort_values(
        [
            "balanced_accuracy",
            "f1",
        ],
        ascending=False,
    ).head(10)
)


Threshold escolhido via OOF temporal: 0.75


,threshold,roc_auc,pr_auc,brier,balanced_accuracy,precision,recall,f1
70,0.7500,0.4332,0.4024,0.3078,0.5101,0.5000,0.1111,0.1818
71,0.7600,0.4332,0.4024,0.3078,0.5101,0.5000,0.1111,0.1818
67,0.7200,0.4332,0.4024,0.3078,0.5025,0.4615,0.1111,0.1791
68,0.7300,0.4332,0.4024,0.3078,0.5025,0.4615,0.1111,0.1791
69,0.7400,0.4332,0.4024,0.3078,0.5025,0.4615,0.1111,0.1791
31,0.3600,0.4332,0.4024,0.3078,0.5021,0.4518,0.5784,0.5073
0,0.0500,0.4332,0.4024,0.3078,0.5000,0.4500,1.0000,0.6207
1,0.0600,0.4332,0.4024,0.3078,0.5000,0.4500,1.0000,0.6207
2,0.0700,0.4332,0.4024,0.3078,0.5000,0.4500,1.0000,0.6207
3,0.0800,0.4332,0.4024,0.3078,0.5000,0.4500,1.0000,0.6207



# Parte C — Refit no desenvolvimento e teste final

Somente agora os modelos vencedores são ajustados com os 40 contratos de desenvolvimento completos e avaliados nos 8 contratos mais recentes.


In [11]:

reg_dev = reg_all[
    reg_all[
        "id_contrato"
    ].isin(
        development_contracts
    )
].copy()

reg_test = reg_all[
    reg_all[
        "id_contrato"
    ].isin(
        test_contracts
    )
].copy()


best_reg_model = (
    regression_models()[
        BEST_REGRESSION_MODEL
    ]
)

best_reg_model.fit(
    reg_dev[
        FEATURES
    ],
    reg_dev[
        "target_dias_atraso"
    ].astype(float),
    model__sample_weight=reg_dev[
        "sample_weight_contrato"
    ].astype(float),
)


reg_test_pred = np.maximum(
    0,
    best_reg_model.predict(
        reg_test[
            FEATURES
        ]
    ),
)


reg_test_metrics = {
    "modelo": BEST_REGRESSION_MODEL,
    "contratos_teste": (
        reg_test[
            "id_contrato"
        ].nunique()
    ),
    **regression_metrics(
        reg_test[
            "target_dias_atraso"
        ].astype(float),
        reg_test_pred,
        reg_test[
            "sample_weight_contrato"
        ].astype(float),
    ),
}


reg_test_results = pd.DataFrame([
    reg_test_metrics
])


display(reg_test_results)


,modelo,contratos_teste,mae,rmse,r2
0,HistGradientBoosting,8,21.8750,54.3519,-0.1933


In [12]:

clf_dev = clf_all[
    clf_all[
        "id_contrato"
    ].isin(
        development_contracts
    )
].copy()

clf_test = clf_all[
    clf_all[
        "id_contrato"
    ].isin(
        test_contracts
    )
].copy()


best_clf_model = (
    classification_models()[
        BEST_CLASSIFICATION_MODEL
    ]
)

best_clf_model.fit(
    clf_dev[
        FEATURES
    ],
    clf_dev[
        "target_atraso"
    ].astype(int),
    model__sample_weight=clf_dev[
        "sample_weight_classificacao"
    ].astype(float),
)


clf_test_prob = (
    best_clf_model.predict_proba(
        clf_test[
            FEATURES
        ]
    )[:, 1]
)


clf_test_metrics = classification_metrics(
    clf_test[
        "target_atraso"
    ].astype(int),
    clf_test_prob,
    threshold=BEST_THRESHOLD,
    sample_weight=clf_test[
        "sample_weight_classificacao"
    ].astype(float),
)


clf_test_results = pd.DataFrame([
    {
        "modelo": BEST_CLASSIFICATION_MODEL,
        "threshold": BEST_THRESHOLD,
        "contratos_teste": (
            clf_test[
                "id_contrato"
            ].nunique()
        ),
        **clf_test_metrics,
    }
])


display(clf_test_results)


clf_test_pred = (
    clf_test_prob
    >= BEST_THRESHOLD
).astype(int)


print(
    "Matriz de confusão por snapshot — teste final:"
)

print(
    confusion_matrix(
        clf_test[
            "target_atraso"
        ].astype(int),
        clf_test_pred,
    )
)


,modelo,threshold,contratos_teste,roc_auc,pr_auc,brier,balanced_accuracy,precision,recall,f1
0,ExtraTrees,0.7500,8,0.4450,0.2373,0.2382,0.5000,0.0000,0.0000,0.0000


Matriz de confusão por snapshot — teste final:
[[121   0]
 [ 16   0]]


## 8. Avaliação final por contrato em horizontes 180/90/60/30 dias

In [13]:

reg_test_eval = reg_test[
    [
        "id_contrato",
        "data_corte",
        "dias_restantes_prazo_vigente",
        "target_dias_atraso",
    ]
].copy()

reg_test_eval[
    "pred_dias_atraso"
] = reg_test_pred


clf_test_eval = clf_test[
    [
        "id_contrato",
        "data_corte",
        "dias_restantes_prazo_vigente",
        "target_atraso",
    ]
].copy()

clf_test_eval[
    "prob_atraso"
] = clf_test_prob

clf_test_eval[
    "pred_atraso"
] = clf_test_pred


def nearest_snapshot(
    data,
    horizon,
):
    subset = data[
        data[
            "dias_restantes_prazo_vigente"
        ].ge(0)
    ].copy()

    subset[
        "dist_horizonte"
    ] = (
        subset[
            "dias_restantes_prazo_vigente"
        ]
        - horizon
    ).abs()

    return (
        subset
        .sort_values(
            [
                "id_contrato",
                "dist_horizonte",
                "data_corte",
            ]
        )
        .groupby(
            "id_contrato",
            as_index=False,
        )
        .first()
    )


reg_horizon_rows = []
clf_horizon_rows = []
detail_90 = None

for horizon in [
    180,
    90,
    60,
    30,
]:
    reg_h = nearest_snapshot(
        reg_test_eval,
        horizon,
    )

    clf_h = nearest_snapshot(
        clf_test_eval,
        horizon,
    )

    reg_horizon_rows.append({
        "horizonte_dias": horizon,
        "contratos": (
            reg_h[
                "id_contrato"
            ].nunique()
        ),
        "mae": mean_absolute_error(
            reg_h[
                "target_dias_atraso"
            ],
            reg_h[
                "pred_dias_atraso"
            ],
        ),
        "rmse": np.sqrt(
            mean_squared_error(
                reg_h[
                    "target_dias_atraso"
                ],
                reg_h[
                    "pred_dias_atraso"
                ],
            )
        ),
    })

    if clf_h[
        "target_atraso"
    ].nunique() > 1:
        auc = roc_auc_score(
            clf_h[
                "target_atraso"
            ],
            clf_h[
                "prob_atraso"
            ],
        )

        ap = average_precision_score(
            clf_h[
                "target_atraso"
            ],
            clf_h[
                "prob_atraso"
            ],
        )

        bac = balanced_accuracy_score(
            clf_h[
                "target_atraso"
            ],
            clf_h[
                "pred_atraso"
            ],
        )

    else:
        auc = np.nan
        ap = np.nan
        bac = np.nan

    clf_horizon_rows.append({
        "horizonte_dias": horizon,
        "contratos": (
            clf_h[
                "id_contrato"
            ].nunique()
        ),
        "taxa_atraso": (
            clf_h[
                "target_atraso"
            ].mean()
        ),
        "roc_auc": auc,
        "pr_auc": ap,
        "balanced_accuracy": bac,
    })

    if horizon == 90:
        detail_90 = clf_h.copy()


reg_horizon_results = pd.DataFrame(
    reg_horizon_rows
)

clf_horizon_results = pd.DataFrame(
    clf_horizon_rows
)


print("Regressão por horizonte:")
display(reg_horizon_results)

print("Classificação por horizonte:")
display(clf_horizon_results)

print("Detalhe de classificação ~90 dias:")
display(
    detail_90[
        [
            "id_contrato",
            "data_corte",
            "dias_restantes_prazo_vigente",
            "target_atraso",
            "prob_atraso",
            "pred_atraso",
        ]
    ]
)


Regressão por horizonte:


,horizonte_dias,contratos,mae,rmse
0,180,8,21.8750,54.3519
1,90,8,21.8750,54.3519
2,60,8,21.8750,54.3519
3,30,8,21.8750,54.3519


Classificação por horizonte:


,horizonte_dias,contratos,taxa_atraso,roc_auc,pr_auc,balanced_accuracy
0,180,8,0.2500,0.8333,0.5833,0.5000
1,90,8,0.2500,0.8333,0.5833,0.5000
2,60,8,0.2500,0.5000,0.3333,0.5000
3,30,8,0.2500,0.4167,0.2917,0.5000


Detalhe de classificação ~90 dias:


,id_contrato,data_corte,dias_restantes_prazo_vigente,target_atraso,prob_atraso,pred_atraso
0,173,2024-07-31,80,0,0.3950,0
1,201,2024-11-30,102,0,0.3883,0
2,254,2025-05-31,83,0,0.5131,0
3,283,2024-11-30,97,0,0.4822,0
4,287,2025-03-31,84,0,0.4691,0
5,317,2024-11-30,84,1,0.5010,0
6,342,2025-07-18,60,0,0.4779,0
7,377,2025-04-29,90,1,0.4842,0


## 9. Importância das features do modelo final quando disponível

In [14]:

feature_importance = pd.DataFrame()


if BEST_CLASSIFICATION_MODEL in {
    "RandomForest",
    "ExtraTrees",
}:
    transformed_names = (
        best_clf_model[
            "preprocessing"
        ]
        .get_feature_names_out()
    )

    importances = (
        best_clf_model[
            "model"
        ]
        .feature_importances_
    )

    feature_importance = pd.DataFrame({
        "feature": transformed_names,
        "importancia": importances,
    }).sort_values(
        "importancia",
        ascending=False,
    )

    display(
        feature_importance.head(20)
    )

else:
    print(
        "HistGradientBoosting não expõe feature_importances_ diretamente; "
        "nenhuma importância foi calculada para evitar usar o teste como base de seleção."
    )


,feature,importancia
19,categorical__classificacao_contrato_Reforma Pa...,0.1343
1,numeric__perc_desconto_modelo,0.1036
0,numeric__prazo_original_dias,0.0878
20,categorical__id_setor_divisao_8,0.0849
4,numeric__pct_dias_aditados_sobre_prazo_original,0.0746
9,numeric__atraso_medio_liberacao_dias,0.0686
23,categorical__id_setor_divisao_32,0.0633
6,numeric__pct_parcelas_atrasadas,0.0586
12,numeric__missingindicator_perc_desconto_modelo,0.0556
21,categorical__id_setor_divisao_10,0.0450


## 10. Exportação

In [15]:

FEATURES_PATH = (
    OUTPUT_DIR
    / "04B_features_14.csv"
)

FOLDS_PATH = (
    OUTPUT_DIR
    / "04B_folds_temporais.csv"
)

REG_CV_PATH = (
    OUTPUT_DIR
    / "04B_regressao_cv_folds.csv"
)

REG_CV_SUMMARY_PATH = (
    OUTPUT_DIR
    / "04B_regressao_cv_resumo.csv"
)

CLF_CV_PATH = (
    OUTPUT_DIR
    / "04B_classificacao_cv_folds.csv"
)

CLF_CV_SUMMARY_PATH = (
    OUTPUT_DIR
    / "04B_classificacao_cv_resumo.csv"
)

OOF_PATH = (
    OUTPUT_DIR
    / "04B_classificacao_oof.csv"
)

THRESHOLD_PATH = (
    OUTPUT_DIR
    / "04B_threshold_oof.csv"
)

REG_TEST_PATH = (
    OUTPUT_DIR
    / "04B_regressao_teste_final.csv"
)

CLF_TEST_PATH = (
    OUTPUT_DIR
    / "04B_classificacao_teste_final.csv"
)

REG_HORIZON_PATH = (
    OUTPUT_DIR
    / "04B_regressao_horizontes_teste.csv"
)

CLF_HORIZON_PATH = (
    OUTPUT_DIR
    / "04B_classificacao_horizontes_teste.csv"
)

DETAIL_90_PATH = (
    OUTPUT_DIR
    / "04B_classificacao_detalhe_90d.csv"
)


pd.DataFrame({
    "feature": FEATURES,
    "tipo": [
        (
            "numerica"
            if feature
            in NUMERIC_FEATURES
            else "categorica"
        )
        for feature in FEATURES
    ],
}).to_csv(
    FEATURES_PATH,
    index=False,
    encoding="utf-8-sig",
)

fold_summary.to_csv(
    FOLDS_PATH,
    index=False,
    encoding="utf-8-sig",
)

reg_cv.to_csv(
    REG_CV_PATH,
    index=False,
    encoding="utf-8-sig",
)

reg_cv_summary.to_csv(
    REG_CV_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

clf_cv.to_csv(
    CLF_CV_PATH,
    index=False,
    encoding="utf-8-sig",
)

clf_cv_summary.to_csv(
    CLF_CV_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

oof_predictions.to_csv(
    OOF_PATH,
    index=False,
    encoding="utf-8-sig",
)

threshold_table.to_csv(
    THRESHOLD_PATH,
    index=False,
    encoding="utf-8-sig",
)

reg_test_results.to_csv(
    REG_TEST_PATH,
    index=False,
    encoding="utf-8-sig",
)

clf_test_results.to_csv(
    CLF_TEST_PATH,
    index=False,
    encoding="utf-8-sig",
)

reg_horizon_results.to_csv(
    REG_HORIZON_PATH,
    index=False,
    encoding="utf-8-sig",
)

clf_horizon_results.to_csv(
    CLF_HORIZON_PATH,
    index=False,
    encoding="utf-8-sig",
)

detail_90.to_csv(
    DETAIL_90_PATH,
    index=False,
    encoding="utf-8-sig",
)

if len(feature_importance):
    feature_importance.to_csv(
        OUTPUT_DIR
        / "04B_importancia_features_classificacao.csv",
        index=False,
        encoding="utf-8-sig",
    )


print("=" * 76)
print("NOTEBOOK 04 — MODELOS NÃO LINEARES + VALIDAÇÃO TEMPORAL")
print("=" * 76)

print(
    "Features:",
    len(FEATURES),
)

print(
    "Folds temporais:",
    len(folds),
)

print(
    "Contratos desenvolvimento:",
    len(development_contracts),
)

print(
    "Contratos teste final:",
    len(test_contracts),
)

print()
print(
    "Melhor regressão:",
    BEST_REGRESSION_MODEL,
)

print(
    "Melhor classificação:",
    BEST_CLASSIFICATION_MODEL,
)

print(
    "Threshold OOF:",
    round(
        BEST_THRESHOLD,
        2,
    ),
)

print()
print(
    "Arquivos:",
    OUTPUT_DIR.resolve(),
)


NOTEBOOK 04 — MODELOS NÃO LINEARES + VALIDAÇÃO TEMPORAL
Features: 14
Folds temporais: 4
Contratos desenvolvimento: 40
Contratos teste final: 8

Melhor regressão: HistGradientBoosting
Melhor classificação: ExtraTrees
Threshold OOF: 0.75

Arquivos: C:\mnt\data\04B_modelos_arvores_features_14_outputs
